___
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRU6ycfMWHLjnsOPma9wwzKXckhE5iYOGp-dA&s" width="380px" height="200px" />


# <font color= #bbc28d> **Halloween Wikipedia Views** </font>
#### <font color= #2E9AFE> `Examen 3 - MNLP`</font>
- <Strong> Diana Valdivia, Sofía Maldonado, Samantha Sánchez & Rafael Takata. </Strong>
- <Strong> Fecha </Strong>: 13/05/2026.

___

<p style="text-align:right;"> Image retrieved from: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRU6ycfMWHLjnsOPma9wwzKXckhE5iYOGp-dA&s</p>

In [6]:
import pandas as pd
import numpy as np
import requests
import joblib
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

tf.random.set_seed(42)
np.random.seed(42)

In [2]:
df_train = pd.read_csv('halloween_train.csv', index_col=0, parse_dates=True)
df_val = pd.read_csv('halloween_val.csv', index_col=0, parse_dates=True)
df_test  = pd.read_csv('halloween_test.csv',  index_col=0, parse_dates=True)

FEATURES = ['views', 'es_halloween', 'semana_halloween', 'dias_para_halloween', 'views_lag7']

df_train = df_train[FEATURES]
df_val = df_val[FEATURES]
df_test  = df_test[FEATURES]

print(f'Train: {df_train.index.min().date()} -> {df_train.index.max().date()} ({len(df_train)} dias)')
print(f'Val: {df_val.index.min().date()} -> {df_val.index.max().date()} ({len(df_val)} dias)')
print(f'Test:  {df_test.index.min().date()}  -> {df_test.index.max().date()}  ({len(df_test)} dias)')

Train: 2020-01-15 -> 2024-09-30 (1721 dias)
Val: 2024-10-01 -> 2024-10-31 (31 dias)
Test:  2025-10-01  -> 2025-10-31  (31 dias)


In [6]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(df_train[FEATURES])
val_scaled   = scaler.transform(df_val[FEATURES])
test_scaled  = scaler.transform(df_test[FEATURES])

# Scaler independiente para views (para invertir predicciones)
scaler_views = MinMaxScaler()
scaler_views.fit(df_train[['views']])

print('Escalado OK')

Escalado OK


In [ ]:
W = 30  # ventana de 30 dias

def make_windows(data, W):
    X, y = [], []
    for i in range(W, len(data)):
        X.append(data[i-W:i])
        y.append(data[i, 0])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Ventanas — val necesita cola del train
cola_val  = train_scaled[-W:]
cola_test = val_scaled[-W:]   # el test toma cola del val, no del train

X_train, y_train = make_windows(train_scaled, W)
X_val,   y_val   = make_windows(np.concatenate([cola_val,  val_scaled]),  W)
X_test,  y_test  = make_windows(np.concatenate([cola_test, test_scaled]), W)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_val: {X_val.shape} | y_val: {y_val.shape}')
print(f'X_test:  {X_test.shape}  | y_test:  {y_test.shape}')

X_train: (1691, 30, 5) | y_train: (1691,)
X_val: (31, 30, 5) | y_val: (31,)
X_test:  (31, 30, 5)  | y_test:  (31,)


In [67]:
X_tr, y_tr = X_train, y_train
N_FEATURES = X_train.shape[2]

In [ ]:
def build_ffnn(window, n_features):
    """Creamos una función que haga un FFNN"""
    model = keras.Sequential([
        layers.Input(shape=(window, n_features)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name='FFNN')
    return model


def build_lstm(n_features, hidden=64, num_layers=2):
    """Creamos una función que sea una LSTM"""
    model = keras.Sequential(name='LSTM')
    model.add(layers.Input(shape=(None, n_features)))
    model.add(layers.LSTM(hidden, return_sequences=False,dropout=0.3, recurrent_dropout=0.2))
    model.add(layers.Dense(16, activation='relu')) 
    model.add(layers.Dropout(0.2))
    model.add(layers.Dense(1))
    return model


ffnn = build_ffnn(W, N_FEATURES)
lstm = build_lstm(N_FEATURES)

print(f'FFNN parametros: {ffnn.count_params():,}')
print(f'LSTM parametros: {lstm.count_params():,}')
ffnn.summary()
lstm.summary()

FFNN parametros: 11,777
LSTM parametros: 18,977


Model: "FFNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_10 (Flatten)            │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 64)             │         9,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,777 (46.00 KB)

 Trainable params: 11,777 (46.00 KB)

...

Model: "LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_20 (LSTM)                  │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,977 (74.13 KB)

 Trainable params: 18,977 (74.13 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
def make_sample_weights(dates_tr, spike_weight=15.0, ramp_weight=3.0):
    weights = np.ones(len(dates_tr))
    for i, d in enumerate(dates_tr):
        if d.month == 10 and d.day == 31:
            weights[i] = spike_weight
        elif d.month == 10 and d.day >= 24:
            weights[i] = ramp_weight
    return weights

def train_model(model, X_tr, y_tr, X_val, y_val, dates_tr,
                epochs=300, lr=1e-3, patience=30, batch_size=32,
                spike_weight=15, ramp_weight=3): 

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='mse'
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=patience,
        restore_best_weights=True,
        verbose=1
    )

    # Reduce lr cuando val_loss se estanca
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,       # divide lr a la mitad
        patience=10,      # si no mejora en 10 épocas
        min_lr=1e-6,
        verbose=1
    )

    log_cb = keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: print(
            f'  Epoca {epoch+1:3d} | train: {logs["loss"]:.6f} | val: {logs["val_loss"]:.6f} | lr: {logs.get("learning_rate", lr):.2e}'
        ) if (epoch + 1) % 5 == 0 else None
    )

    sample_weights = make_sample_weights(dates_tr, spike_weight=spike_weight, ramp_weight=ramp_weight)

    history = model.fit(
        X_tr, y_tr,
        sample_weight=sample_weights,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop, reduce_lr, log_cb],
        verbose=0
    )

    return history.history['loss'], history.history['val_loss']

dates_tr = df_train.index[W:]   # fechas que corresponden a X_train

print('=== Entrenando FFNN ===')
ffnn = build_ffnn(W, N_FEATURES)
h_tr_ffnn, h_val_ffnn = train_model(ffnn, X_tr, y_tr, X_val, y_val, dates_tr)

print('\n=== Entrenando LSTM ===')
lstm = build_lstm(N_FEATURES)
h_tr_lstm, h_val_lstm = train_model(lstm, X_tr, y_tr, X_val, y_val, dates_tr, spike_weight=5, ramp_weight=1.5)

=== Entrenando FFNN ===
  Epoca   5 | train: 0.005335 | val: 0.007740 | lr: 1.00e-03
  Epoca  10 | train: 0.003912 | val: 0.005899 | lr: 1.00e-03
  Epoca  15 | train: 0.002864 | val: 0.003490 | lr: 1.00e-03
  Epoca  20 | train: 0.001558 | val: 0.002693 | lr: 1.00e-03
  Epoca  25 | train: 0.003677 | val: 0.003470 | lr: 1.00e-03
  Epoca  30 | train: 0.003024 | val: 0.002372 | lr: 1.00e-03
  Epoca  35 | train: 0.001238 | val: 0.001640 | lr: 1.00e-03
  Epoca  40 | train: 0.001360 | val: 0.003906 | lr: 1.00e-03
  Epoca  45 | train: 0.001710 | val: 0.002529 | lr: 1.00e-03
  Epoca  50 | train: 0.001421 | val: 0.002402 | lr: 1.00e-03
  Epoca  55 | train: 0.001408 | val: 0.001241 | lr: 1.00e-03

Epoch 58: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
  Epoca  60 | train: 0.001047 | val: 0.001435 | lr: 5.00e-04
  Epoca  65 | train: 0.000909 | val: 0.000848 | lr: 5.00e-04

Epoch 68: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
  Epoca  70 | train: 0.0003

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['FFNN — Curva de perdida', 'LSTM — Curva de perdida'])

for col, (h_tr, h_val) in enumerate(
    [(h_tr_ffnn, h_val_ffnn), (h_tr_lstm, h_val_lstm)], start=1
):
    fig.add_trace(go.Scatter(y=h_tr,  name='Train',
                             line=dict(color='#5B6547'), showlegend=(col==1)), row=1, col=col)
    fig.add_trace(go.Scatter(y=h_val, name='Validacion',
                             line=dict(color="#385C6E", dash='dash'), showlegend=(col==1)), row=1, col=col)

fig.update_layout(template='plotly_white', height=350,
                  title='Curvas de perdida (MSE normalizado)')
fig.update_xaxes(title_text='Epoca')
fig.update_yaxes(title_text='MSE')
fig.write_html('Assets/curvas_perdida.html',
               include_plotlyjs='cdn', 
               full_html=False)
fig.show()

In [56]:
def recursive_forecast(model, seed_window, exog_future_scaled, steps):
    """
    seed_window        : (W, n_features) ultimos W dias reales de train
    exog_future_scaled : (steps, n_features-1) variables exogenas del test ya escaladas
    """
    window = seed_window.copy()
    preds  = []

    for i in range(steps):
        # Keras espera (batch, timesteps, features) -> unsqueeze con np.expand_dims
        x = np.expand_dims(window, axis=0)          # (1, W, n_features)
        pred_scaled = model.predict(x, verbose=0)[0, 0]
        preds.append(pred_scaled)

        new_row = np.array([pred_scaled] + list(exog_future_scaled[i]), dtype=np.float32)
        window  = np.vstack([window[1:], new_row])

    return np.array(preds)


seed      = train_scaled[-W:]
exog_test = test_scaled[:, 1:]   # columnas exogenas (sin views)
steps     = len(df_test)

preds_ffnn_sc = recursive_forecast(ffnn, seed, exog_test, steps)
preds_lstm_sc = recursive_forecast(lstm, seed, exog_test, steps)

preds_ffnn = scaler_views.inverse_transform(preds_ffnn_sc.reshape(-1,1)).flatten()
preds_lstm = scaler_views.inverse_transform(preds_lstm_sc.reshape(-1,1)).flatten()
real       = df_test['views'].values

print('Forecast generado OK')

Forecast generado OK


In [72]:
fechas = df_test.index

fig = go.Figure()
fig.add_trace(go.Scatter(x=fechas, y=real,
                         name='Real', line=dict(color='black', width=2)))
fig.add_trace(go.Scatter(x=fechas, y=preds_ffnn,
                         name='FFNN', line=dict(color="#95a565", dash='dash', width=2)))
fig.add_trace(go.Scatter(x=fechas, y=preds_lstm,
                         name='LSTM', line=dict(color='#2a9d8f', dash='dot', width=2)))

fig.add_vline(
    x=pd.Timestamp('2025-10-31').timestamp() * 1000,
    line=dict(color='black', dash='dash', width=1),
    annotation_text='Halloween 2025'
)

fig.update_layout(
    title='Forecast — Octubre 2025 | FFNN vs LSTM',
    xaxis_title='Fecha', yaxis_title='Views',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.write_html('Assets/forecast_recursivo.html',
               include_plotlyjs='cdn', 
               full_html=False)
fig.show()

In [58]:
def compute_metrics(real, preds, nombre):
    mae  = mean_absolute_error(real, preds)
    rmse = np.sqrt(mean_squared_error(real, preds))
    mape = np.mean(np.abs((real - preds) / (real + 1))) * 100
    print(f'{nombre:6s} -> MAE: {mae:>10,.0f} | RMSE: {rmse:>10,.0f} | MAPE: {mape:.1f}%')
    return {'Modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape}

m_ffnn = compute_metrics(real, preds_ffnn, 'FFNN')
m_lstm = compute_metrics(real, preds_lstm, 'LSTM')

metrics_df = pd.DataFrame([m_ffnn, m_lstm]).set_index('Modelo').round(2)
metrics_df

FFNN   -> MAE:      5,802 | RMSE:     13,204 | MAPE: 13.8%
LSTM   -> MAE:     11,091 | RMSE:     25,535 | MAPE: 31.0%


,MAE,RMSE,MAPE (%)
Modelo,,,
FFNN,5802.07,13204.48,13.76
LSTM,11091.30,25534.65,30.97


In [75]:
fig = make_subplots(rows=1, cols=3, subplot_titles=['MAE', 'RMSE', 'MAPE (%)'])
colores = ['#95a565', '#2a9d8f']
modelos = ['FFNN', 'LSTM']

for col, metrica in enumerate(['MAE', 'RMSE', 'MAPE (%)'], start=1):
    vals = [m_ffnn[metrica], m_lstm[metrica]]
    fig.add_trace(
        go.Bar(x=modelos, y=vals, marker_color=colores,
               text=[f'{v:,.1f}' for v in vals],
               textposition='outside', showlegend=False),
        row=1, col=col
    )

fig.update_layout(title='Comparacion de metricas — FFNN vs LSTM',
                  template='plotly_white', height=500)
fig.write_html('Assets/metricas_comparativa.html',
               include_plotlyjs='cdn', 
               full_html=False)
fig.show()

In [76]:
errores_ffnn = np.abs(real - preds_ffnn)
errores_lstm = np.abs(real - preds_lstm)

fig = go.Figure()
fig.add_trace(go.Bar(x=fechas, y=errores_ffnn,
                     name='Error FFNN', marker_color='#95a565', opacity=0.8))
fig.add_trace(go.Bar(x=fechas, y=errores_lstm,
                     name='Error LSTM', marker_color='#2a9d8f', opacity=0.8))

fig.add_vline(
    x=pd.Timestamp('2025-10-31').timestamp() * 1000,
    line=dict(color='black', dash='dash'),
    annotation_text='Halloween'
)

fig.update_layout(
    title='Error absoluto por dia — donde falla cada modelo?',
    xaxis_title='Fecha', yaxis_title='Error absoluto (views)',
    barmode='group', template='plotly_white'
)
fig.write_html('Assets/errores_acumulados.html',
               include_plotlyjs='cdn', 
               full_html=False)
fig.show()

spike_idx  = fechas.day == 31
normal_idx = ~spike_idx

print('Error en el spike (31-oct 2025):')
print(f'  FFNN: {errores_ffnn[spike_idx][0]:,.0f} views')
print(f'  LSTM: {errores_lstm[spike_idx][0]:,.0f} views')

print('\nError promedio dias normales (1-30 oct):')
print(f'  FFNN: {errores_ffnn[normal_idx].mean():,.0f} views')
print(f'  LSTM: {errores_lstm[normal_idx].mean():,.0f} views')

Error en el spike (31-oct 2025):
  FFNN: 51,338 views
  LSTM: 14,011 views

Error promedio dias normales (1-30 oct):
  FFNN: 4,284 views
  LSTM: 10,994 views


In [78]:
preds_ensemble = (preds_ffnn + preds_lstm) / 2
m_ens = compute_metrics(real, preds_ensemble, 'Ensemble')

# Grafica con ensemble
fechas = df_test.index

fig = go.Figure()
fig.add_trace(go.Scatter(x=fechas, y=real,
                         name='Real', line=dict(color='black', width=2)))
fig.add_trace(go.Scatter(x=fechas, y=preds_ffnn,
                         name='FFNN', line=dict(color='#95a565', dash='dash', width=2)))
fig.add_trace(go.Scatter(x=fechas, y=preds_lstm,
                         name='LSTM', line=dict(color='#2a9d8f', dash='dot', width=2)))
fig.add_trace(go.Scatter(x=fechas, y=preds_ensemble,
                         name='Ensemble', line=dict(color="#875f96", width=2)))

fig.add_vline(
    x=pd.Timestamp('2025-10-31').timestamp() * 1000,
    line=dict(color='black', dash='dash', width=1),
    annotation_text='Halloween 2025'
)

fig.update_layout(
    title='Forecast — Octubre 2025 | FFNN vs LSTM vs Ensemble',
    xaxis_title='Fecha', yaxis_title='Views',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.write_html('Assets/forecast_ensamble.html',
               include_plotlyjs='cdn', 
               full_html=False)
fig.show()

# Errores por dia
errores_ffnn     = np.abs(real - preds_ffnn)
errores_lstm     = np.abs(real - preds_lstm)
errores_ensemble = np.abs(real - preds_ensemble)

fig = go.Figure()
fig.add_trace(go.Bar(x=fechas, y=errores_ffnn,
                     name='Error FFNN', marker_color='#95a565', opacity=0.8))
fig.add_trace(go.Bar(x=fechas, y=errores_lstm,
                     name='Error LSTM', marker_color='#2a9d8f', opacity=0.8))
fig.add_trace(go.Bar(x=fechas, y=errores_ensemble,
                     name='Error Ensemble', marker_color='#9b59b6', opacity=0.8))

fig.add_vline(
    x=pd.Timestamp('2025-10-31').timestamp() * 1000,
    line=dict(color='black', dash='dash'),
    annotation_text='Halloween'
)

fig.update_layout(
    title='Error absoluto por dia — FFNN vs LSTM vs Ensemble',
    xaxis_title='Fecha', yaxis_title='Error absoluto (views)',
    barmode='group', template='plotly_white'
)
fig.write_html('Assets/errores_ensamble.html',
               include_plotlyjs='cdn', 
               full_html=False)
fig.show()

# Errores spike vs normales
spike_idx  = fechas.day == 31
normal_idx = ~spike_idx

print('Error en el spike (31-oct 2025):')
print(f'  FFNN:     {errores_ffnn[spike_idx][0]:,.0f} views')
print(f'  LSTM:     {errores_lstm[spike_idx][0]:,.0f} views')
print(f'  Ensemble: {errores_ensemble[spike_idx][0]:,.0f} views')

print('\nError promedio dias normales (1-30 oct):')
print(f'  FFNN:     {errores_ffnn[normal_idx].mean():,.0f} views')
print(f'  LSTM:     {errores_lstm[normal_idx].mean():,.0f} views')
print(f'  Ensemble: {errores_ensemble[normal_idx].mean():,.0f} views')

Ensemble -> MAE:      8,062 | RMSE:     18,314 | MAPE: 19.4%


Error en el spike (31-oct 2025):
  FFNN:     51,338 views
  LSTM:     14,011 views
  Ensemble: 32,674 views

Error promedio dias normales (1-30 oct):
  FFNN:     4,284 views
  LSTM:     10,994 views
  Ensemble: 7,242 views


In [ ]:
import joblib

# joblib.dump(scaler,        'Models/Scaler/scaler.pkl')
# joblib.dump(scaler_views,  'Models/Scaler/scaler_views.pkl')

# ffnn.save('Models/ffnn_halloween.keras')
# lstm.save('Models/lstm_halloween.keras')

print('Scalers guardados OK')
print('Modelos guardados OK')

Scalers guardados OK
Modelos guardados OK


In [14]:
ffnn         = keras.models.load_model('Models/ffnn_halloween.keras')
lstm         = keras.models.load_model('Models/lstm_halloween.keras')
scaler       = joblib.load('Models/Scaler/scaler.pkl')
scaler_views = joblib.load('Models/Scaler/scaler_views.pkl')

W = 30
FEATURES = ['views', 'es_halloween', 'semana_halloween', 'dias_para_halloween', 'views_lag7']

url = (
    "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
    "en.wikipedia/all-access/user/Halloween/daily/20260101/20260512"
)
r = requests.get(url, headers={"User-Agent": "halloween-forecast/1.0"})
items = r.json()["items"]

df_fresh = pd.DataFrame(items)
df_fresh["date"] = pd.to_datetime(df_fresh["timestamp"], format="%Y%m%d%H")
df_fresh = df_fresh.sort_values("date").set_index("date")[["views"]]
df_fresh

,views
date,
2026-01-01,1797
2026-01-02,1379
2026-01-03,4218
2026-01-04,5029
2026-01-05,1603
...,...
2026-05-07,1252
2026-05-08,1151
2026-05-09,1061


In [15]:
def days_to_halloween(date):
    hw = pd.Timestamp(date.year, 10, 31)
    diff = (hw - date).days
    return diff if diff >= 0 else (pd.Timestamp(date.year + 1, 10, 31) - date).days

df_fresh['es_halloween']        = ((df_fresh.index.month == 10) & (df_fresh.index.day == 31)).astype(int)
df_fresh['semana_halloween']    = ((df_fresh.index.month == 10) & (df_fresh.index.day >= 24)).astype(int)
df_fresh['dias_para_halloween'] = df_fresh.index.map(days_to_halloween)
df_fresh['views_lag7']          = df_fresh['views'].shift(7)
df_fresh = df_fresh.dropna()[FEATURES]

print(f"Datos frescos: {df_fresh.index.min().date()} → {df_fresh.index.max().date()} ({len(df_fresh)} días)")

Datos frescos: 2026-01-08 → 2026-05-11 (124 días)


In [16]:
fresh_scaled = scaler.transform(df_fresh)
seed = fresh_scaled[-W:]  # últimos 30 días reales como ventana inicial

In [24]:
fecha_inicio = pd.Timestamp('2026-05-13')
fecha_fin    = pd.Timestamp('2027-10-31')
fechas_pred  = pd.date_range(fecha_inicio, fecha_fin, freq='D')

# Usar los últimos 7 días reales para los primeros lags
views_reales = df_fresh['views'].values

exog_rows = []
for i, d in enumerate(fechas_pred):
    # valores reales históricos
    if i < 7:
        lag7 = float(views_reales[-(7 - i)])
    else:
        lag7 = 0.0  # placeholder, se sobreescribe

    exog_rows.append({
        'views':               0.0,
        'es_halloween':        int(d.month == 10 and d.day == 31),
        'semana_halloween':    int(d.month == 10 and d.day >= 24),
        'dias_para_halloween': float(days_to_halloween(d)),
        'views_lag7':          lag7
    })

df_exog = pd.DataFrame(exog_rows, index=fechas_pred)
df_exog

,views,es_halloween,semana_halloween,dias_para_halloween,views_lag7
2026-05-13,0.0,0,0,171.0,1236.0
2026-05-14,0.0,0,0,170.0,1265.0
2026-05-15,0.0,0,0,169.0,1252.0
2026-05-16,0.0,0,0,168.0,1151.0
2026-05-17,0.0,0,0,167.0,1061.0
...,...,...,...,...,...
2027-10-27,0.0,0,1,4.0,0.0
2027-10-28,0.0,0,1,3.0,0.0
2027-10-29,0.0,0,1,2.0,0.0
2027-10-30,0.0,0,1,1.0,0.0


In [25]:
def recursive_forecast_exog(model, seed_window, df_exog_raw, scaler, scaler_views, steps):
    window = seed_window.copy()
    preds  = []

    for i in range(steps):
        x       = np.expand_dims(window, axis=0)
        pred_sc = model.predict(x, verbose=0)[0, 0]
        preds.append(pred_sc)
        
        row_raw          = df_exog_raw.iloc[i].astype(float).copy()
        row_raw['views'] = scaler_views.inverse_transform([[pred_sc]])[0, 0]

        if i >= 7:
            row_raw['views_lag7'] = scaler_views.inverse_transform([[preds[i-7]]])[0, 0]

        row_df     = pd.DataFrame([row_raw], columns=FEATURES)
        row_scaled = scaler.transform(row_df)[0]
        window     = np.vstack([window[1:], row_scaled])

    return np.array(preds)


steps         = len(fechas_pred)
preds_ffnn_sc = recursive_forecast_exog(ffnn, seed, df_exog, scaler, scaler_views, steps)
preds_lstm_sc = recursive_forecast_exog(lstm, seed, df_exog, scaler, scaler_views, steps)

preds_ffnn = scaler_views.inverse_transform(preds_ffnn_sc.reshape(-1,1)).flatten()
preds_lstm = scaler_views.inverse_transform(preds_lstm_sc.reshape(-1,1)).flatten()
preds_ens  = (preds_ffnn + preds_lstm) / 2

In [34]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_fresh.index, y=df_fresh['views'],
    name='Real (histórico)', line=dict(color='black', width=1.5)))

fig.add_trace(go.Scatter(
    x=fechas_pred, y=preds_ffnn,
    name='FFNN', line=dict(color='#95a565', width=2)))

fig.add_trace(go.Scatter(
    x=fechas_pred, y=preds_lstm,
    name='LSTM', line=dict(color='#2a9d8f', dash='dash', width=2)))

fig.add_trace(go.Scatter(
    x=fechas_pred, y=preds_ens,
    name='Ensamble', line=dict(color="#956ea4", dash='dot', width=2)))

fig.add_vline(
    x=pd.Timestamp('2026-10-31').timestamp() * 1000,
    line=dict(color='black', dash='dash', width=1),
    annotation_text='Halloween 2026'
)
fig.add_vline(
    x=pd.Timestamp('2027-10-31').timestamp() * 1000,
    line=dict(color='black', dash='dash', width=1),
    annotation_text='Halloween 2027'
)
fig.add_vline(
    x=pd.Timestamp('2026-05-12').timestamp() * 1000,
    line=dict(color='gray', dash='dot', width=1),
    annotation_text='Hoy'
)

fig.update_layout(
    title='Forecast Halloween — Mayo a Octubre 2027',
    xaxis_title='Fecha', yaxis_title='Views',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.write_html('Assets/predicciones_halloween.html',
               include_plotlyjs='cdn', 
               full_html=False)
fig.show()

In [33]:
print(f"\nPredicción Halloween 2026 (31-oct):")
idx_31 = fechas_pred.get_loc(pd.Timestamp('2026-10-31'))
print(f"  FFNN:     {preds_ffnn[idx_31]:>10,.1f} views")
print(f"  LSTM:     {preds_lstm[idx_31]:>10,.1f} views")
print(f"  Ensamble: {preds_ens[idx_31]:>10,.1f} views")
print(f"\nPredicción Halloween 2027 (31-oct):")
idx_31 = fechas_pred.get_loc(pd.Timestamp('2027-10-31'))
print(f"  FFNN:     {preds_ffnn[idx_31]:>10,.1f} views")
print(f"  LSTM:     {preds_lstm[idx_31]:>10,.1f} views")
print(f"  Ensamble: {preds_ens[idx_31]:>10,.1f} views")


Predicción Halloween 2026 (31-oct):
  FFNN:      334,735.2 views
  LSTM:      282,703.9 views
  Ensamble:  308,719.6 views

Predicción Halloween 2027 (31-oct):
  FFNN:      334,735.9 views
  LSTM:      282,703.9 views
  Ensamble:  308,719.9 views
